# NB01 — Verify Data

**Runs locally — no GPU needed.**

Checks:
- DOP20 patch dimensions and channel count (should be RGBI → 4 channels, we use first 3)
- Preview a sample patch
- Print the class prompt lists from our fork

Attach no Kaggle datasets. Run this before every Kaggle session to confirm your local data is intact.

In [ ]:
import sys
from pathlib import Path

# Add src/ to path so segearth_utils is importable
ROOT = Path("__file__").parent.parent if "__file__" in dir() else Path(".").resolve().parent
sys.path.insert(0, str(ROOT / "src"))

from segearth_utils.constants import HESSEN_CLASSES, HESSEN_COLOR_MAP, POTSDAM_CLASSES
print("segearth_utils imported OK")

In [ ]:
import numpy as np
from PIL import Image
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

DATA_DIR = ROOT / "data"
TILE_EXTS = [".tif", ".tiff", ".png", ".jpg", ".jpeg"]

patches = sorted(p for ext in TILE_EXTS for p in DATA_DIR.rglob(f"*{ext}"))
print(f"Found {len(patches)} patch file(s) under {DATA_DIR}")
for p in patches[:5]:
    print(" ", p.relative_to(ROOT))

In [ ]:
# Inspect the first patch — check channels, size, dtype
if patches:
    img = Image.open(patches[0])
    arr = np.array(img)
    print(f"File:    {patches[0].name}")
    print(f"Mode:    {img.mode}")
    print(f"Shape:   {arr.shape}  (H x W x C)")
    print(f"Dtype:   {arr.dtype}")
    print(f"Min/Max: {arr.min()} / {arr.max()}")
    if arr.ndim == 3 and arr.shape[2] == 4:
        print("\nDOP20 is RGBI (4 channels). For SAM3 we use only the first 3 (RGB).")
        rgb = arr[:, :, :3]
    elif arr.ndim == 3 and arr.shape[2] == 3:
        print("\nAlready RGB (3 channels).")
        rgb = arr
    else:
        print("Unexpected shape — check data source.")
        rgb = None
else:
    print("No patches found. Copy some DOP20 .tif files into data/ to verify.")
    rgb = None

In [ ]:
# Preview patch + color legend
if rgb is not None:
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    axes[0].imshow(rgb)
    axes[0].set_title(f"RGB preview: {patches[0].name}")
    axes[0].axis("off")

    legend_img = np.zeros((len(HESSEN_CLASSES) * 40, 200, 3), dtype=np.uint8)
    for i, (cls, col) in enumerate(zip(HESSEN_CLASSES, HESSEN_COLOR_MAP)):
        legend_img[i*40:(i+1)*40] = col
    axes[1].imshow(legend_img)
    axes[1].set_yticks([i*40+20 for i in range(len(HESSEN_CLASSES))])
    axes[1].set_yticklabels(HESSEN_CLASSES)
    axes[1].set_xticks([])
    axes[1].set_title("Hessen class colors")

    plt.tight_layout()
    out = ROOT / "results" / "nb01_preview.png"
    plt.savefig(out, dpi=100, bbox_inches="tight")
    plt.show()
    print(f"Saved: {out}")

In [ ]:
print("=== Hessen 6-class prompts (cls_hessen.txt) ===")
for i, cls in enumerate(HESSEN_CLASSES):
    print(f"  [{i}] {cls}")

print()
print("=== Potsdam 6-class prompts (cls_potsdam.txt) ===")
for i, cls in enumerate(POTSDAM_CLASSES):
    print(f"  [{i}] {cls}")